In [2]:
import sys
sys.path.insert(0, "..")


from src.preprocessing.preprocessing import PreProcessing
from src.tokenization.tokenizer import BPETokenizer

In [3]:
preprocessor = PreProcessing("../dataset/tiny-stories/train.csv")
preprocessor.load_data().sample_corpus(n=100_000, random_state=42)

print(f"Full dataset: {len(preprocessor.data):,} stories")
print(f"Corpus size: {len(preprocessor.corpus):,} characters ({len(preprocessor.corpus) / 1e6:.1f} MB)")
print(f"\nFirst 300 characters:\n{preprocessor.corpus[:300]}")

Full dataset: 2,119,719 stories
Corpus size: 90,143,813 characters (90.1 MB)

First 300 characters:
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine with her friends. One day, Lily's mom told her they needed to go to the store to buy some food for dinner. Lily didn't like going to the store because it was boring, but she didn't worry too much.

As the


In [4]:
tokenizer = BPETokenizer(vocab_size=4096, special_tokens=["<EOS>"])
tokenizer.train(preprocessor.corpus, verbose=True)

Merge 100/3839: (277, 286) -> 356 (' day', count=127,924)
Merge 200/3839: (117, 114) -> 456 ('ur', count=56,217)
Merge 300/3839: (319, 110) -> 556 ('own', count=32,755)
Merge 400/3839: (314, 115) -> 656 ('ess', count=22,062)
Merge 500/3839: (97, 335) -> 756 ('ace', count=16,141)
Merge 600/3839: (299, 111) -> 856 (' So', count=12,588)
Merge 700/3839: (381, 796) -> 956 (' share', count=9,969)
Merge 800/3839: (596, 107) -> 1056 (' walk', count=8,100)
Merge 900/3839: (632, 373) -> 1156 (' Mommy', count=6,953)
Merge 1000/3839: (260, 289) -> 1256 (' sing', count=5,812)
Merge 1100/3839: (65, 115) -> 1356 ('As', count=5,123)
Merge 1200/3839: (908, 115) -> 1456 (' books', count=4,456)
Merge 1300/3839: (853, 288) -> 1556 ('This', count=3,746)
Merge 1400/3839: (381, 287) -> 1656 (' shar', count=3,321)
Merge 1500/3839: (295, 794) -> 1756 (' nap', count=2,927)
Merge 1600/3839: (1720, 385) -> 1856 (' welcome', count=2,565)
Merge 1700/3839: (1192, 463) -> 1956 ('ountain', count=2,313)
Merge 1800/3839

In [5]:
print(f"Vocabulary size: {len(tokenizer.vocab)}")
print(f"Number of merges: {len(tokenizer.merges)}")

def display_token(t):
    return t.decode("utf-8", errors="replace") if isinstance(t, bytes) else t

print("\nFirst 20 merges (most common byte pairs):")
for i, merge in enumerate(tokenizer.merges[:20]):
    merged = tokenizer.vocab[257 + i]
    print(f"  {i}: '{display_token(tokenizer.vocab[merge[0]])}' + "
          f"'{display_token(tokenizer.vocab[merge[1]])}' -> '{display_token(merged)}'")

print(f"\nLast 20 merges (longest learned tokens):")
for i in range(max(0, len(tokenizer.merges) - 20), len(tokenizer.merges)):
    merge = tokenizer.merges[i]
    merged = tokenizer.vocab[257 + i]
    print(f"  {i}: '{display_token(tokenizer.vocab[merge[0]])}' + "
          f"'{display_token(tokenizer.vocab[merge[1]])}' -> '{display_token(merged)}'")

Vocabulary size: 4096
Number of merges: 3839

First 20 merges (most common byte pairs):
  0: 'h' + 'e' -> 'he'
  1: ' ' + 't' -> ' t'
  2: ' ' + 'a' -> ' a'
  3: ' ' + 's' -> ' s'
  4: ' ' + 'w' -> ' w'
  5: 'n' + 'd' -> 'nd'
  6: ' t' + 'he' -> ' the'
  7: 'e' + 'd' -> 'ed'
  8: ' a' + 'nd' -> ' and'
  9: ' t' + 'o' -> ' to'
  10: ' ' + 'b' -> ' b'
  11: 'i' + 'n' -> 'in'
  12: ' ' + 'h' -> ' h'
  13: ' w' + 'a' -> ' wa'
  14: 'r' + 'e' -> 're'
  15: 'o' + 'u' -> 'ou'
  16: ' ' + 'f' -> ' f'
  17: 'i' + 't' -> 'it'
  18: ' ' + 'c' -> ' c'
  19: ' ' + 'l' -> ' l'

Last 20 merges (longest learned tokens):
  3819: ' climb' + 'ing' -> ' climbing'
  3820: ' tr' + 'ump' -> ' trump'
  3821: ' me' + 'at' -> ' meat'
  3822: ' b' + 'ur' -> ' bur'
  3823: ' Joe' + 'y' -> ' Joey'
  3824: ' c' + 'elery' -> ' celery'
  3825: ' spl' + 'it' -> ' split'
  3826: ' c' + 'ord' -> ' cord'
  3827: ' p' + 'ed' -> ' ped'
  3828: ' listen' + 'ing' -> ' listening'
  3829: ' ph' + 'ot' -> ' phot'
  3830: ' h' +

In [6]:
test_texts = [
    "Once upon a time, there was a little girl.",
    "The cat sat on the mat.",
    "Hello<EOS>World",
    'She said, "I love you!"',
]

for text in test_texts:
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)
    print(f"Original:   {text!r}")
    print(f"Token IDs:  {ids}")
    print(f"Decoded:    {decoded!r}")
    print(f"Num tokens: {len(ids)}")
    print(f"Roundtrip:  {'OK' if text == decoded else 'MISMATCH'}")
    print()

Original:   'Once upon a time, there was a little girl.'
Token IDs:  [431, 447, 259, 396, 44, 399, 282, 259, 398, 442, 46]
Decoded:    'Once upon a time, there was a little girl.'
Num tokens: 11
Roundtrip:  OK

Original:   'The cat sat on the mat.'
Token IDs:  [372, 691, 1411, 346, 263, 1926, 46]
Decoded:    'The cat sat on the mat.'
Num tokens: 7
Roundtrip:  OK

Original:   'Hello<EOS>World'
Token IDs:  [1401, 256, 87, 302, 324]
Decoded:    'Hello<EOS>World'
Num tokens: 5
Roundtrip:  OK

Original:   'She said, "I love you!"'
Token IDs:  [1075, 327, 44, 328, 73, 819, 348, 424]
Decoded:    'She said, "I love you!"'
Num tokens: 8
Roundtrip:  OK



In [8]:
# Compression ratio on held-out stories
eval_preprocessor = PreProcessing("../dataset/tiny-stories/train.csv")
eval_preprocessor.load_data().sample_corpus(n=1_000, random_state=99)

encoded = tokenizer.encode(eval_preprocessor.corpus)
num_bytes = len(eval_preprocessor.corpus.encode("utf-8"))
num_tokens = len(encoded)
ratio = num_bytes / num_tokens

print(f"Eval corpus: {num_bytes:,} bytes")
print(f"Encoded: {num_tokens:,} tokens")
print(f"Compression ratio: {ratio:.2f} bytes/token")

Eval corpus: 909,490 bytes
Encoded: 229,915 tokens
Compression ratio: 3.96 bytes/token


In [9]:
tokenizer.save("../dataset/tokenizer.json")
print("Tokenizer saved to dataset/tokenizer.json")

Tokenizer saved to dataset/tokenizer.json


In [1]:
loaded = BPETokenizer.load("../dataset/tokenizer.json")

test = "Once upon a time<EOS>"
assert loaded.encode(test) == tokenizer.encode(test)
assert loaded.decode(loaded.encode(test)) == test
print("Load/save roundtrip verified!")

In [ ]:
from src.tokenization.corpus_encoder import CorpusEncoder

tokenizer = BPETokenizer.load("../dataset/tokenizer.json")
encoder = CorpusEncoder(tokenizer, parts_dir="../dataset/tokens_parts")

# Encode in parallel — resumes from already-saved parts
# 8 workers on a 10-core machine (leaves 2 cores for system/Jupyter)
encoder.encode_parts(
    csv_path="../dataset/tiny-stories/train.csv",
    stories_per_chunk=5_000,
    n_workers=8,
)

Loading stories...
2,119,489 stories -> 424 parts of 5,000

Resuming — 49/424 parts already done, skipping

  [50/424] part_00050 — 1,078,492 tokens — 244s elapsed — ~91392s remaining
  [51/424] part_00056 — 1,046,132 tokens — 252s elapsed — ~46953s remaining
  [52/424] part_00053 — 1,067,709 tokens — 255s elapsed — ~31648s remaining
  [53/424] part_00052 — 1,157,325 tokens — 272s elapsed — ~25205s remaining
  [54/424] part_00055 — 1,130,309 tokens — 274s elapsed — ~20263s remaining
  [55/424] part_00051 — 1,217,903 tokens — 279s elapsed — ~17187s remaining
  [56/424] part_00054 — 1,238,130 tokens — 287s elapsed — ~15064s remaining
  [57/424] part_00049 — 1,270,570 tokens — 298s elapsed — ~13648s remaining
  [58/424] part_00058 — 1,138,095 tokens — 528s elapsed — ~21464s remaining
  [59/424] part_00057 — 1,194,502 tokens — 533s elapsed — ~19468s remaining
  [60/424] part_00061 — 1,134,743 tokens — 546s elapsed — ~18077s remaining
  [61/424] part_00059 — 1,241,966 tokens — 553s elapsed 

In [ ]:
# Run this after all parts are encoded
tokens = encoder.merge_parts("../dataset/tokens.npy")

In [ ]:
print("Done")

In [ ]:
0